# KNN Classifier — Hotel Booking Cancellation Prediction

Author: Siriwardana N.D.V.S

KNN (K-Nearest Neighbors) is a supervised, non-parametric, instance-based lazy learner. Unlike other models, it stores the entire training set and classifies new bookings by majority vote among the K most similar training examples using Euclidean or Manhattan distance.

The objective of this notebook is to predict whether a hotel booking will be cancelled (`is_canceled = 1`) or not (`is_canceled = 0`) using the Hotel Booking Demand dataset, and compare KNN performance against the group's other models (Logistic Regression, Decision Tree, Random Forest).

In [ ]:
import os
import sys

current_dir = os.path.abspath(os.getcwd())
project_root = None

for _ in range(6):
    config_path = os.path.join(current_dir, "src", "config.py")
    if os.path.exists(config_path):
        project_root = current_dir
        break
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

if project_root is None:
    raise RuntimeError(
        "Could not find src/config.py within 5 parent levels. "
        "Open VS Code from the project root and rerun this cell."
    )

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
import sklearn

from src import config
from src.data_loader import load_hotel_bookings, basic_train_ready_checks
from src.preprocessing import build_preprocessor, PreprocessOptions, get_feature_names
from src.train_eval import (
    TrainOptions, split_xy, make_train_test_split,
    get_estimator, build_model_pipeline, tune_with_gridsearch,
    predict_with_optional_proba, evaluate_on_test
    )
from src.metrics import compute_classification_metrics, format_metrics_for_print
from src.plots import plot_confusion_matrix, plot_roc_curve, plot_pr_curve
from src.io_utils import (ensure_artifact_dirs, save_json, save_text,
                          save_dataframe, save_model, save_run_metadata)

DIRS = ensure_artifact_dirs()
print("Artifact directories:")
for name, path in DIRS.items():
    print(f"- {name}: {path}")

print(f"Python version: {sys.version}")
print(f"scikit-learn version: {sklearn.__version__}")